# Qwen2.5-7B-Instruct — **BASE MODEL** Değerlendirme Raporu

Bu notebook, **fine-tune edilmemiş** orijinal `Qwen/Qwen2.5-7B-Instruct` modelini
`qwen2_test_updated.jsonl` test seti üzerinde değerlendirir.

Amaç: Fine-tune etkisini kanıtlamak için **baseline (taban)** değerler elde etmek.
Sonuçları `02_evaluate_qwen25_7b.ipynb` çıktısıyla karşılaştırabilirsiniz.

| Parametre | Değer | Açıklama |
|---|---|---|
| Model | `Qwen/Qwen2.5-7B-Instruct` | HuggingFace — fine-tune YOK |
| INFERENCE_BATCH | 16 | 16 kayıt aynı anda → ~16x hız |
| dtype | bfloat16 | Fine-tune notebook ile aynı hassasiyet |
| MAX_NEW_TOKENS | 1536 | Yeterli çıktı uzunluğu |
| Tahmini süre | ~5-8 dk | 522 kayıt, A100 80GB |

## Hücre 1: Kurulum
⚠️ Çalıştırdıktan sonra **Runtime → Restart Runtime** yapın!

In [ ]:
%%capture
!pip install -q -U unsloth
!pip install -q -U --no-deps trl peft accelerate bitsandbytes
print('Kurulum tamamlandı. Runtime → Restart Runtime yapın!')

## Hücre 2: Google Drive Bağlantısı ve Yol Ayarları

> ⚠️ `DATA_DIR` değişkenini kendi Drive klasörünüzün yoluyla güncelleyin.
> Test dosyasının (`qwen2_test_updated.jsonl`) bu klasörde bulunması gerekiyor.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ================================================================
# BURAYA KENDI KLASOR ADINIZI YAZIN (01_finetune ile AYNI):
DATA_DIR  = "/content/drive/MyDrive/qwen2_egitim_seti"
# ================================================================

TEST_PATH = os.path.join(DATA_DIR, "qwen2_test_updated.jsonl")

exists = os.path.exists(TEST_PATH)
print(f"{'✅' if exists else '❌'} TEST: {'BULUNDU' if exists else 'BULUNAMADI — yolu kontrol edin!'}")
print(f"   {TEST_PATH}")

## Hücre 3: Base Modeli Yükle (HuggingFace — BF16, Fine-Tune YOK)

Fine-tuned notebook'ta `MODEL_PATH` yerine doğrudan `Qwen/Qwen2.5-7B-Instruct` kullanıyoruz.
Unsloth aynı chat template'i desteklediğinden prompt formatı birebir aynı kalıyor.

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 3072
BASE_MODEL_ID  = "Qwen/Qwen2.5-7B-Instruct"  # Fine-tune YOK — orijinal model

print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Model yükleniyor: {BASE_MODEL_ID} ...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = torch.bfloat16,
    load_in_4bit   = False,
)
FastLanguageModel.for_inference(model)  # Unsloth 2x hızlı inference modu

# Batched inference için sol padding
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"\n✅ BASE MODEL yüklendi ve inference moduna alındı!")
print(f"   Model: {BASE_MODEL_ID}")

## Hücre 4: Test Setini Yükle

In [ ]:
import json

test_records = []
with open(TEST_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            test_records.append(json.loads(line))

print(f"✅ Test seti: {len(test_records)} kayıt yüklendi.")

## Hücre 5: Fonksiyonlar — Batched Inference + JSON Parse + Kural Kontrol

Bu fonksiyonlar `02_evaluate_qwen25_7b.ipynb` ile **birebir aynı** — fair comparison için.

**Kritik optimizasyon:** Tek tek değil, 16 kayıt **aynı anda** GPU'ya gönderiyor.
A100 80GB'da 16 paralel inference, tekli inference'a göre **~12-15x daha hızlı**.

In [ ]:
import re

REQUIRED_KEYS = [
    "performed_actions", "target_unit", "response_type",
    "process_status", "result_information", "process_information", "draft",
]

INFERENCE_BATCH = 16  # A100 80GB: 16 kayıt aynı anda
MAX_NEW_TOKENS  = 1536

# ── Batched inference ────────────────────────────────────────────
def generate_batch(batch_messages_list):
    """batch_messages_list: [{role, content}, ...] listelerinden oluşan liste."""
    prompts = [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        for msgs in batch_messages_list
    ]
    inputs = tokenizer(
        prompts,
        return_tensors      = "pt",
        padding             = True,
        truncation          = True,
        max_length          = MAX_SEQ_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens  = MAX_NEW_TOKENS,
            do_sample       = False,   # Deterministik değerlendirme
            temperature     = 1.0,
            use_cache       = True,
            pad_token_id    = tokenizer.eos_token_id,
        )

    # Sadece yeni üretilen token'ları decode et
    input_len  = inputs["input_ids"].shape[1]
    new_tokens = output_ids[:, input_len:]
    return tokenizer.batch_decode(new_tokens, skip_special_tokens=True)


# ── JSON parse ──────────────────────────────────────────────────
def try_parse_json(raw_text):
    text = raw_text.strip()
    text = re.sub(r"^```(json)?", "", text).strip()
    text = re.sub(r"```$",        "", text).strip()
    try:
        return json.loads(text)
    except Exception:
        start, end = text.find("{"), text.rfind("}")
        if start != -1 and end != -1 and end > start:
            try:
                return json.loads(text[start:end + 1])
            except Exception:
                return None
        return None


# ── Kural kontrolleri ───────────────────────────────────────────
CLOSING_RICA     = ("rica ederim", "rica ederiz")
CLOSING_ARZ      = ("arz ederim", "arz ederiz")
CLOSING_ARZ_RICA = ("arz ve rica ederim", "arz ve rica ederiz")

def check_closing(draft: str, sender_type: str) -> bool:
    lower = draft.lower()
    has_rica  = any(p in lower for p in CLOSING_RICA)
    has_arz   = any(p in lower for p in CLOSING_ARZ)
    has_combo = any(p in lower for p in CLOSING_ARZ_RICA)
    st = (sender_type or "").upper()
    if st in ("VATANDAS", "OZEL_KURULUS"):
        return has_rica and not has_arz
    elif st == "KAMU_KURUMU":
        return has_combo
    else:
        return has_arz and not has_rica

def check_legislation_citation(draft: str, legislation: list) -> bool:
    if not legislation:
        return True
    for leg in legislation:
        law_num = re.search(r"\d+", leg.get("law_name") or "")
        art_num = re.search(r"\d+", leg.get("article")  or "")
        if (law_num and law_num.group(0) in draft) or (art_num and art_num.group(0) in draft):
            return True
    return False

def check_min_words(draft: str, min_words: int = 180) -> bool:
    return len(draft.split()) >= min_words

print(f"✅ Fonksiyonlar tanımlandı. Inference batch boyutu: {INFERENCE_BATCH}")

## Hücre 6: Batched Değerlendirme Döngüsü
**A100 80GB'da 522 kayıt → ~5-8 dakika** (batch=16, base model biraz daha yavaş olabilir)

In [ ]:
from tqdm.auto import tqdm
import math

LIMIT = None  # Hızlı test için 32, tüm test seti için None

records_to_eval = test_records[:LIMIT] if LIMIT else test_records
n_batches = math.ceil(len(records_to_eval) / INFERENCE_BATCH)
print(f"Toplam kayıt : {len(records_to_eval)}")
print(f"Batch sayısı : {n_batches} (batch başına {INFERENCE_BATCH} kayıt)")
print("Başlıyor...\n")

results = []

for batch_start in tqdm(range(0, len(records_to_eval), INFERENCE_BATCH), desc="Batch"):
    batch = records_to_eval[batch_start : batch_start + INFERENCE_BATCH]

    # Her örnek için system + user mesajlarını hazırla
    batch_msgs   = []
    batch_meta   = []
    for rec in batch:
        msgs_dict    = {m["role"]: m["content"] for m in rec["messages"]}
        user_data    = json.loads(msgs_dict["user"])
        gold         = json.loads(msgs_dict["assistant"])
        batch_msgs.append([
            {"role": "system", "content": msgs_dict["system"]},
            {"role": "user",   "content": msgs_dict["user"]},
        ])
        batch_meta.append({
            "sender_type" : user_data.get("sender_type"),
            "legislation" : user_data.get("selected_legislation") or [],
            "gold"        : gold,
        })

    # Batch inference — 16 kayıt aynı anda GPU'ya
    raw_outputs = generate_batch(batch_msgs)

    for meta, raw_output in zip(batch_meta, raw_outputs):
        results.append({
            **meta,
            "pred"       : try_parse_json(raw_output),
            "raw_output" : raw_output,
        })

print(f"\n✅ {len(results)} kayıt değerlendirildi.")

## Hücre 7: Metrikleri Hesapla ve Raporu Yazdır

Metrik hesaplama `02_evaluate_qwen25_7b.ipynb` ile **birebir aynı** — adil karşılaştırma için.

In [ ]:
n = len(results)

json_valid             = 0
json_valid_complete    = 0
target_unit_correct    = 0
response_type_correct  = 0
process_status_correct = 0
closing_correct        = 0
legislation_faithful   = 0
min_words_ok           = 0

for row in results:
    pred = row["pred"]
    gold = row["gold"]

    if pred is not None:
        json_valid += 1
        if all(k in pred and str(pred[k]).strip() not in ("", "[]") for k in REQUIRED_KEYS):
            json_valid_complete += 1

        if str(pred.get("target_unit", "")).strip().lower() == str(gold.get("target_unit", "")).strip().lower():
            target_unit_correct += 1
        if pred.get("response_type") == gold.get("response_type"):
            response_type_correct += 1
        if pred.get("process_status") == gold.get("process_status"):
            process_status_correct += 1

        draft = str(pred.get("draft", ""))
        if draft:
            if check_closing(draft, row["sender_type"]):
                closing_correct += 1
            if check_legislation_citation(draft, row["legislation"]):
                legislation_faithful += 1
            if check_min_words(draft):
                min_words_ok += 1

def pct(x, total=n):
    return round(100 * x / total, 2) if total else 0.0

report = {
    "toplam_kayit":              n,
    "json_gecerlilik_orani_%":   pct(json_valid),
    "json_tam_alanli_orani_%":   pct(json_valid_complete),
    "target_unit_dogruluk_%":    pct(target_unit_correct),
    "response_type_dogruluk_%":  pct(response_type_correct),
    "process_status_dogruluk_%": pct(process_status_correct),
    "kapanis_kural_uyum_%":      pct(closing_correct),
    "mevzuat_sadakati_%":        pct(legislation_faithful),
    "min_kelime_uyum_%":         pct(min_words_ok),
}

weights = {
    "json_tam_alanli_orani_%":   0.15,
    "target_unit_dogruluk_%":    0.20,
    "response_type_dogruluk_%":  0.15,
    "process_status_dogruluk_%": 0.15,
    "kapanis_kural_uyum_%":      0.15,
    "mevzuat_sadakati_%":        0.20,
}
report["GENEL_DOGRULUK_SKORU_%"] = round(sum(report[k] * w for k, w in weights.items()), 2)

print("=" * 60)
print("  QWEN2.5-7B BASE MODEL DEĞERLENDİRME RAPORU")
print("  (Fine-tune edilmemiş — Baseline)")
print("=" * 60)
for k, v in report.items():
    print(f"{k:35s}: {v}")
print("=" * 60)

## Hücre 8: Fine-Tune vs Base Model Karşılaştırma Tablosu

Fine-tune notebook (`02_evaluate_qwen25_7b.ipynb`) değerlerini buraya elle girin
ve yan yana karşılaştırma tablosunu görün.

In [ ]:
# ================================================================
# 02_evaluate_qwen25_7b.ipynb çıktısındaki değerleri buraya girin:
finetuned_report = {
    "json_gecerlilik_orani_%":   None,  # örn: 98.5
    "json_tam_alanli_orani_%":   None,
    "target_unit_dogruluk_%":    None,
    "response_type_dogruluk_%":  None,
    "process_status_dogruluk_%": None,
    "kapanis_kural_uyum_%":      None,
    "mevzuat_sadakati_%":        None,
    "min_kelime_uyum_%":         None,
    "GENEL_DOGRULUK_SKORU_%":    None,
}
# ================================================================

metrics = [k for k in finetuned_report]
print(f"{"Metrik":<35} {"Base Model":>12} {"Fine-Tuned":>12} {"Fark":>10}")
print("-" * 72)
for k in metrics:
    base_val = report.get(k)
    ft_val   = finetuned_report.get(k)
    if base_val is not None and ft_val is not None:
        diff = ft_val - base_val
        diff_str = f"+{diff:.2f}" if diff >= 0 else f"{diff:.2f}"
    else:
        diff_str = "—"
        ft_val   = ft_val if ft_val is not None else "—"
    print(f"{k:<35} {str(base_val):>12} {str(ft_val):>12} {diff_str:>10}")
print("-" * 72)

## Hücre 9: (Opsiyonel) Hatalı Örnekleri İncele

In [ ]:
hatali = [
    r for r in results
    if r["pred"] is None
    or r["pred"].get("response_type") != r["gold"].get("response_type")
][:5]

for r in hatali:
    print("-" * 50)
    print(f"GOLD response_type : {r['gold'].get('response_type')}")
    print(f"PRED response_type : {(r['pred'] or {}).get('response_type')}")
    print(f"GOLD target_unit   : {r['gold'].get('target_unit')}")
    print(f"PRED target_unit   : {(r['pred'] or {}).get('target_unit')}")
    print(f"RAW (ilk 500 kr)   : {r['raw_output'][:500]}")

## Hücre 10: Raporu Drive'a Kaydet

In [ ]:
rapor_yolu = os.path.join(DATA_DIR, "base_model_degerlendirme_raporu.json")
with open(rapor_yolu, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
print(f"✅ Base model raporu kaydedildi: {rapor_yolu}")